# 07 - Regional comparison

This notebook asks whether vulnerability is spread evenly across Israel. The earlier steps built the graph at the station level, counted its articulation points, and scored each station with centrality measures. Here we aggregate those per-station results by region (north / center / south / Jerusalem, assigned by coordinates back in step 01) and by metro area (Tel Aviv / Haifa / Jerusalem / Be'er Sheva / periphery), and compare how concentrated the critical infrastructure is in each. The output is one comparison table plus the figures that go into the report.

**The research question answered here:** *Does the damage from removing key stations fall differently between the center of the country and the periphery?*

### Input (produced in earlier notebooks)
- `outputs/nb/04_centrality_analysis/tables/stop_metrics.csv` - one row per station with
  `stop_id, stop_name, region, metro, lat, lon, degree, betweenness, ...`
- `outputs/nb/03_network_descriptive_analysis/tables/articulation_points.csv` - the stations whose
  removal disconnects the graph
- `outputs/nb/03_network_descriptive_analysis/tables/bridges.csv` - *optional*; if it isn't there,
  the bridge columns are simply skipped

The folder names aren't hard-coded: the loader scans the `outputs/nb` tree for the file names and prefers a folder whose name starts with the expected step number.

### Output (all under `outputs/nb/07_regional_comparison/`)
`tables/`
- `regional_summary.csv` - the main product: one row per region
- `metro_summary.csv` - the same breakdown by metro area
- `stops_with_region.csv` - every station with the `is_ap` / `is_critical` flags used here
- `top_critical_by_region.csv` - the 5 critical stations with the highest betweenness in each region
- `regional_significance.csv` - a chi-square test of region against criticality

`figures/`
- `critical_stations_by_region.png`, `ap_and_bridges_by_region.png`,
  `avg_betweenness_by_region.png`, `stations_map_by_region.png`,
  `regional_vulnerability_comparison.png`, `metro_vulnerability_comparison.png`

**Runtime:** seconds. This notebook only aggregates CSV files that were already computed in earlier steps - no graph algorithm runs here.

## 1. Set up the working environment

Locates the repository (and clones it when we're on Google Colab), sets the repo root as the working directory, and creates the shared output folder `outputs/nb`. Every notebook in this project opens with this same cell, so the whole series runs unchanged both locally and on Colab.

In [ ]:
# --- Environment bootstrap (safe to re-run, works locally and on Google Colab) ---
import os, sys, subprocess
from pathlib import Path

def _ensure(*pkgs):
    """Install only the packages that are actually missing."""
    import importlib.util
    alias = {"scikit-learn": "sklearn", "python-louvain": "community",
             "python-bidi": "bidi", "node2vec": "node2vec"}
    missing = [p for p in pkgs
               if importlib.util.find_spec(alias.get(p, p.replace("-", "_"))) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

def find_repo_root():
    """Find the repo locally; on Colab, clone it."""
    here = Path(os.getcwd()).resolve()
    for cand in [here, *here.parents]:
        if (cand / "israel-public-transportation").is_dir():
            return cand
    target = Path("/content/israel-transit-network-resilience")
    if not target.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/seanfourman/israel-transit-network-resilience.git",
                        str(target)], check=True)
    return target

REPO = find_repo_root()
os.chdir(REPO)
DATA = REPO / "israel-public-transportation"
OUT = REPO / "outputs" / "nb"
OUT.mkdir(parents=True, exist_ok=True)
print("Repo root:", REPO)

## 2. Libraries, tunable constants, and the step's output folder

We install and import the scientific-computing packages, then declare all the tuning parameters of the analysis in one place, so a reviewer can change the definition of "critical" and rerun:

- `CRITICAL_QUANTILE = 0.90` - the top-decile (top-10%) threshold on betweenness, used throughout the
  project.
- `MIN_REGION_N = 100` - any group with fewer stations than this is flagged as a small sample, since a
  percentage computed on a handful of stations is noise, not a finding.
- `WILSON_Z = 1.96` - the z value for the 95% confidence interval drawn on every proportion.

The step writes only into its own folder, `outputs/nb/07_regional_comparison/`.

In [ ]:
_ensure("pandas", "numpy", "matplotlib", "seaborn", "scipy")

import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", font_scale=1.05)
pd.set_option("display.width", 160)

# ---- Tunables -------------------------------------------------------------------
CRITICAL_QUANTILE = 0.90   # "critical" = betweenness in the top 10% of the whole network
MIN_REGION_N      = 100    # below this many stops, a group is flagged as small-sample
WILSON_Z          = 1.96   # 95% confidence interval on every reported share

# ---- Stage folders --------------------------------------------------------------
STAGE   = OUT / "07_regional_comparison"
TABLES  = STAGE / "tables"
FIGURES = STAGE / "figures"
TABLES.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

print("Stage folder:", STAGE)

## 3. Hebrew text in the figures

The GTFS feed is Israeli, so the station names, region names, and metro names are in Hebrew. Matplotlib doesn't apply the Unicode bidirectional algorithm, so Hebrew comes out reversed. We patch `matplotlib.text.Text.set_text` once, before any plotting. Region and metro names are also translated to English for the figures (see the label maps below), but any value we didn't anticipate falls back to Hebrew in correct character order instead of gibberish.

In [ ]:
# Stop names are Hebrew. Matplotlib does not apply the Unicode bidi algorithm, so
# Hebrew labels render reversed. Patch it once, before drawing any figure.
_ensure("python-bidi")
import re
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.text as mtext
from bidi.algorithm import get_display

_HEBREW_RE = re.compile(r"[\u0590-\u05FF]")

def fix_he(text):
    """Return display-ordered text. Non-Hebrew is returned untouched."""
    if not isinstance(text, str) or not _HEBREW_RE.search(text):
        return text
    return get_display(text)

def install_hebrew():
    # Arial exists on Windows; DejaVu Sans ships with matplotlib and covers Hebrew.
    matplotlib.rcParams["font.family"] = ["Arial", "DejaVu Sans"]
    matplotlib.rcParams["axes.unicode_minus"] = False
    if getattr(mtext.Text, "_bidi_patched", False):
        return
    _orig = mtext.Text.set_text
    def set_text(self, s):
        if isinstance(s, str) and getattr(self, "_bidi_display", None) == s:
            return _orig(self, s)
        fixed = fix_he(s)
        if isinstance(fixed, str):
            self._bidi_display = fixed
        return _orig(self, fixed)
    mtext.Text.set_text = set_text
    mtext.Text._bidi_patched = True

install_hebrew()

## 4. Consistent labels, colors, and order

A small but important detail for a report: the same region has to get the same color in every figure. The original script colored the bars by *position*, so after sorting a region swapped color between charts - we fix that with an explicit name-to-color mapping. Regions are always drawn in the fixed order center, north, south, Jerusalem; any unexpected value is appended at the end in gray.

In [ ]:
REGION_ORDER = ["מרכז", "צפון", "דרום", "ירושלים"]
REGION_EN = {"מרכז": "Center", "צפון": "North",
             "דרום": "South", "ירושלים": "Jerusalem"}
REGION_COLORS = {"מרכז": "#2563eb", "צפון": "#16a34a",
                 "דרום": "#dc2626", "ירושלים": "#d97706"}

METRO_EN = {"תל אביב": "Tel Aviv", "חיפה": "Haifa", "ירושלים": "Jerusalem",
            "באר שבע": "Be'er Sheva", "פריפריה": "Periphery"}
METRO_COLORS = {"תל אביב": "#2563eb", "חיפה": "#0891b2", "ירושלים": "#d97706",
                "באר שבע": "#dc2626", "פריפריה": "#6b7280"}

GREY = "#6b7280"

def region_label(name):
    """English label for a region, falling back to display-ordered Hebrew."""
    return REGION_EN.get(name, fix_he(name))

def metro_label(name):
    return METRO_EN.get(name, fix_he(name))

def region_color(name):
    return REGION_COLORS.get(name, GREY)

def metro_color(name):
    return METRO_COLORS.get(name, GREY)

def in_region_order(values):
    """Sort region names into the canonical order; unknown names go last, alphabetically."""
    known = [r for r in REGION_ORDER if r in set(values)]
    rest = sorted(v for v in set(values) if v not in REGION_ORDER)
    return known + rest

def annotate_bars(ax, bars, texts, pad=0.02, fontsize=9):
    """Write a short label (we use the group size n) just above each bar."""
    top = ax.get_ylim()[1]
    for bar, txt in zip(bars, texts):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + top * pad,
                txt, ha="center", va="bottom", fontsize=fontsize, color="#334155")

print("label helpers ready")

## 5. Load the products from earlier steps

This step consumes results and doesn't recompute them: the station-level centrality table comes from notebook 04, and the articulation-point list from notebook 03. `find_artifact` scans the whole `outputs/nb` tree for a file name and prefers the folder whose name starts with the expected step number, so a rename in an earlier step produces a helpful message instead of a `FileNotFoundError` deep in the analysis. Our own step folder is excluded from the search, so a rerun can never read its own output.

`bridges.csv` is treated as optional - it only adds one descriptive column.

In [ ]:
def find_artifact(filename, prefer_prefix, produced_by):
    """Locate a file written by an earlier notebook stage under outputs/nb."""
    candidates = sorted({p for p in OUT.rglob(filename) if STAGE not in p.parents})
    if not candidates:
        raise FileNotFoundError(
            f"{filename} not found anywhere under {OUT} - "
            f"run notebook {produced_by} first, it writes this file.")
    preferred = [p for p in candidates
                 if p.relative_to(OUT).parts[0].startswith(prefer_prefix)]
    chosen = (preferred or candidates)[0]
    if len(candidates) > 1:
        print(f"  note: found {len(candidates)} copies of {filename}; using {chosen}")
    return chosen

METRICS_PATH = find_artifact("stop_metrics.csv", "04", "04_centrality_analysis")
AP_PATH = find_artifact("articulation_points.csv", "03", "03_network_descriptive_analysis")

metrics = pd.read_csv(METRICS_PATH, dtype={"stop_id": str}, encoding="utf-8-sig")
ap_df = pd.read_csv(AP_PATH, dtype={"stop_id": str}, encoding="utf-8-sig")

try:
    BRIDGES_PATH = find_artifact("bridges.csv", "03", "03_network_descriptive_analysis")
    bridges = pd.read_csv(BRIDGES_PATH,
                          dtype={"from_stop": str, "to_stop": str},
                          encoding="utf-8-sig")
except FileNotFoundError:
    BRIDGES_PATH, bridges = None, None
    print("  bridges.csv not available - the bridge columns will be skipped.")

# Notebook 04 exports the sampled estimate under the name `approx_betweenness`
# (the name records that it is a k-sample estimate, not exact betweenness).
# Alias it so this notebook works with either schema.
if "betweenness" not in metrics.columns and "approx_betweenness" in metrics.columns:
    metrics["betweenness"] = metrics["approx_betweenness"]
    print("  note: aliased 'approx_betweenness' -> 'betweenness' (sampled estimate)")

# Fail loudly and early if the upstream schema is not what we expect.
for col in ("stop_id", "region", "metro", "degree", "betweenness", "lat", "lon"):
    if col not in metrics.columns:
        raise KeyError(f"'{col}' missing from {METRICS_PATH.name}. "
                       f"Columns present: {list(metrics.columns)}")

for col in ("degree", "betweenness", "lat", "lon"):
    metrics[col] = pd.to_numeric(metrics[col], errors="coerce")
metrics["degree"] = metrics["degree"].fillna(0)
metrics["betweenness"] = metrics["betweenness"].fillna(0)
for col in ("region", "metro"):
    metrics[col] = metrics[col].fillna("").replace("", "Unknown")

print(f"stops loaded      : {len(metrics):,}  <- {METRICS_PATH}")
print(f"articulation pts  : {len(ap_df):,}  <- {AP_PATH}")
print(f"bridges           : {0 if bridges is None else len(bridges):,}")
print("stops per region  :")
print(metrics["region"].value_counts().to_string())

## 6. The definition of "critical" used here

This is the most important methodological choice in the notebook, so we state it explicitly.

> **A station is *critical* if its betweenness centrality is in the top decile (top 10%) of the whole network**
> (`betweenness >= p90`, where the threshold is computed once over **all** stations, not separately per region).

This is exactly the definition used in **notebook 04**, where that same top-10% betweenness group is checked for whether it has a walking alternative. Keeping the definition identical means the counts in this notebook line up with the counts there.

Two deliberate consequences:

- **The threshold is global.** If each region had its own p90, every region would be critical in exactly 10% of
  its stations by definition and the comparison would be meaningless. A global threshold is what lets us say
  "region X holds more than its fair share of the network's critical stations".
- **Articulation points are reported separately and not merged in.** An articulation point is a *structural*
  failure point (removing it disconnects the graph), which is a different and stricter notion than heavy traffic.
  The original script OR-ed the two together; we keep the main flag `is_critical` matching notebook 04 and expose
  the union in a secondary, explicitly named column, `is_critical_broad`, so both readings are available without
  confusing them.

One caveat we check in code: betweenness is computed on the largest connected component only, so isolated stations get a score of 0. If more than 10% of stations are tied at the threshold value, the `>=` comparison quietly selects more than 10% of the network - the cell prints the actual share so this is visible.

In [ ]:
ap_set = set(ap_df["stop_id"].astype(str))
metrics["is_ap"] = metrics["stop_id"].astype(str).isin(ap_set)

BTW_THRESHOLD = float(metrics["betweenness"].quantile(CRITICAL_QUANTILE))
metrics["is_critical"] = metrics["betweenness"] >= BTW_THRESHOLD
metrics["is_critical_broad"] = metrics["is_critical"] | metrics["is_ap"]

n_total = len(metrics)
n_crit = int(metrics["is_critical"].sum())
n_ap = int(metrics["is_ap"].sum())
n_both = int((metrics["is_critical"] & metrics["is_ap"]).sum())

print(f"betweenness p{CRITICAL_QUANTILE*100:.0f} threshold : {BTW_THRESHOLD:.8f}")
print(f"critical (top betweenness)   : {n_crit:,}  ({100*n_crit/n_total:.1f}% of all stops)")
print(f"articulation points matched  : {n_ap:,}  ({100*n_ap/n_total:.1f}%)")
print(f"both critical and AP         : {n_both:,}")
print(f"union (is_critical_broad)    : {int(metrics['is_critical_broad'].sum()):,}")

if BTW_THRESHOLD <= 0:
    print("\nWARNING: the p90 threshold is 0, meaning more than 10% of stops have zero "
          "betweenness. Every zero-betweenness stop is being counted as critical - "
          "raise CRITICAL_QUANTILE or restrict to the largest component before trusting this.")
if n_ap == 0:
    print("\nWARNING: no articulation point matched a stop_id in stop_metrics.csv - "
          "the two upstream files may use different id formats.")

## 7. Bridges touching each region

A *bridge* is an edge whose removal disconnects the graph - the edge-level counterpart of an articulation point, and a direct measure of "this region depends on a single line". The original script loaded `bridges.csv` and never used it; here we actually use it.

The counting rule: each bridge is counted **once for every distinct region it touches**. A bridge internal to one region adds 1 to that region; a bridge crossing a regional boundary adds 1 to each of the two. Endpoints whose station isn't in the metrics table (which shouldn't happen) are dropped. The cell does nothing if `bridges.csv` isn't found.

In [ ]:
bridge_counts = None
if bridges is not None and len(bridges) and {"from_stop", "to_stop"} <= set(bridges.columns):
    region_of = metrics.drop_duplicates("stop_id").set_index("stop_id")["region"]
    touched = []
    for a, b in zip(bridges["from_stop"].map(region_of), bridges["to_stop"].map(region_of)):
        touched.extend({r for r in (a, b) if isinstance(r, str)})
    bridge_counts = pd.Series(touched, dtype="object").value_counts()
    print("bridges touching each region:")
    print(bridge_counts.to_string())
else:
    print("no bridge data - skipping the bridge columns")

## 8. The regional summary table

The core of the aggregation. For each region we report the raw counts (`total_stops`, `critical_stops`, `ap_stops`), the averages (`avg_degree`, `avg_betweenness`, `max_betweenness`), and most useful for comparison - the relative **rates**: a region with 13 thousand stations and a region with 3.7 thousand stations can't be compared on counts alone.

Two additions relative to the original script:

- **A 95% Wilson confidence interval** on `pct_critical`. The Wilson interval behaves sensibly for small groups
  and for rates near 0 or 100%, where the standard normal interval doesn't. This is the measure that tells us
  whether two regions really differ or just look different.
- **A `small_sample` flag** (`total_stops < MIN_REGION_N`). Every flagged row is printed as a warning and marked
  with a hatch in the figures; its percentage shouldn't be cited in the report.

The same helper function is reused for the metro breakdown in the next cell.

In [ ]:
def wilson_ci(k, n, z=WILSON_Z):
    """95% Wilson score interval (in percent) for k successes out of n."""
    if n == 0:
        return (np.nan, np.nan)
    p = k / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    half = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / denom
    return (100 * max(0.0, center - half), 100 * min(1.0, center + half))

def summarize(df, key):
    """Aggregate stop-level flags into one row per group (region or metro)."""
    s = df.groupby(key).agg(
        total_stops=("stop_id", "count"),
        critical_stops=("is_critical", "sum"),
        critical_broad_stops=("is_critical_broad", "sum"),
        ap_stops=("is_ap", "sum"),
        avg_degree=("degree", "mean"),
        avg_betweenness=("betweenness", "mean"),
        max_betweenness=("betweenness", "max"),
    ).reset_index()
    for c in ("critical_stops", "critical_broad_stops", "ap_stops"):
        s[c] = s[c].astype(int)
    s["pct_critical"] = (100 * s["critical_stops"] / s["total_stops"]).round(2)
    s["pct_critical_broad"] = (100 * s["critical_broad_stops"] / s["total_stops"]).round(2)
    s["pct_ap"] = (100 * s["ap_stops"] / s["total_stops"]).round(2)
    # share of the network's critical stops that sit in this group
    s["share_of_all_critical"] = (100 * s["critical_stops"] / max(1, n_crit)).round(1)
    s["share_of_all_stops"] = (100 * s["total_stops"] / n_total).round(1)
    ci = [wilson_ci(k, n) for k, n in zip(s["critical_stops"], s["total_stops"])]
    s["pct_critical_lo"] = [round(lo, 2) for lo, _ in ci]
    s["pct_critical_hi"] = [round(hi, 2) for _, hi in ci]
    s["small_sample"] = s["total_stops"] < MIN_REGION_N
    s["avg_degree"] = s["avg_degree"].round(3)
    s["avg_betweenness"] = s["avg_betweenness"].round(8)
    return s.sort_values("pct_critical", ascending=False).reset_index(drop=True)

region_summary = summarize(metrics, "region")
if bridge_counts is not None:
    region_summary["bridges_touching"] = (region_summary["region"]
                                          .map(bridge_counts).fillna(0).astype(int))
    region_summary["bridges_per_1000_stops"] = (
        1000 * region_summary["bridges_touching"] / region_summary["total_stops"]).round(2)

region_summary.to_csv(TABLES / "regional_summary.csv", index=False, encoding="utf-8-sig")
metrics.to_csv(TABLES / "stops_with_region.csv", index=False, encoding="utf-8-sig")

print(region_summary.to_string(index=False))

flagged = region_summary[region_summary["small_sample"]]
if len(flagged):
    print(f"\nSMALL SAMPLE (< {MIN_REGION_N} stops) - do not quote these percentages:")
    print(flagged[["region", "total_stops", "pct_critical"]].to_string(index=False))
else:
    print(f"\nNo region falls below {MIN_REGION_N} stops - all regional shares are on "
          "thousands of stops and are statistically stable.")
print(f"\nsaved -> {TABLES / 'regional_summary.csv'}")

## 9. The same breakdown by metro area

The four regions are geographic strips cut by latitude, which lumps a dense city together with the farmland around it. The `metro` label from step 01 gives a complementary view: stations within a fixed radius of Tel Aviv / Haifa / Jerusalem / Be'er Sheva, and everything else tagged as periphery. Here the small-sample flag may actually fire, so we print it again.

In [ ]:
metro_summary = summarize(metrics, "metro")
metro_summary.to_csv(TABLES / "metro_summary.csv", index=False, encoding="utf-8-sig")
print(metro_summary.to_string(index=False))

flagged_metro = metro_summary[metro_summary["small_sample"]]
if len(flagged_metro):
    print(f"\nSMALL SAMPLE (< {MIN_REGION_N} stops):")
    print(flagged_metro[["metro", "total_stops", "pct_critical",
                         "pct_critical_lo", "pct_critical_hi"]].to_string(index=False))
else:
    print(f"\nNo metro area falls below {MIN_REGION_N} stops.")
print(f"\nsaved -> {TABLES / 'metro_summary.csv'}")

## 10. Is the regional difference real, or does it just look real?

Bar charts always look different. A chi-square test of independence on the two-way table *region x is_critical* asks whether the share of critical stations depends on region at all, and **Cramer's V** turns the chi-square into an effect size between 0 and 1 that doesn't grow with sample size.

Both should be read together and skeptically: with tens of thousands of stations, *any* difference comes out "highly significant" (p here is essentially a function of n). Cramer's V and the confidence intervals from the previous cell are the honest measures of how much the regions actually differ.

In [ ]:
from scipy.stats import chi2_contingency

contingency = pd.crosstab(metrics["region"], metrics["is_critical"])
chi2, p_value, dof, expected = chi2_contingency(contingency)
n_obs = int(contingency.values.sum())
cramers_v = float(np.sqrt(chi2 / (n_obs * (min(contingency.shape) - 1))))
spread = float(region_summary["pct_critical"].max() - region_summary["pct_critical"].min())

sig = pd.DataFrame([{
    "test": "chi-square independence (region x is_critical)",
    "chi2": round(chi2, 2),
    "dof": int(dof),
    "p_value": p_value,
    "n": n_obs,
    "cramers_v": round(cramers_v, 4),
    "pct_critical_spread_points": round(spread, 2),
    "min_expected_count": round(float(expected.min()), 1),
}])
sig.to_csv(TABLES / "regional_significance.csv", index=False, encoding="utf-8-sig")
print(sig.T.to_string(header=False))

print("\nInterpretation:")
print(f"  p = {p_value:.3g} -> the regions do differ, but at n={n_obs:,} that was almost "
      "guaranteed.")
print(f"  Cramer's V = {cramers_v:.3f} -> "
      + ("negligible" if cramers_v < 0.1 else "small" if cramers_v < 0.3
         else "moderate" if cramers_v < 0.5 else "large") + " association.")
print(f"  spread between the highest and lowest region: {spread:.1f} percentage points.")

## 11. Figure 1 - critical stations by region

Two panels, because they answer two different questions. **Left**: the absolute number of critical stations, dominated by the region that simply contains the most stations. **Right**: the *share* of each region's own stations that are critical, with a 95% Wilson interval as the error bar - this is the panel that answers "is the periphery more fragile?". The group size `n` is printed above each bar, and a small-sample region would be drawn with a hatch.

In [ ]:
reg = region_summary.set_index("region").loc[in_region_order(region_summary["region"])].reset_index()
labels = [region_label(r) for r in reg["region"]]
colors = [region_color(r) for r in reg["region"]]
hatches = ["//" if flag else "" for flag in reg["small_sample"]]
n_texts = [f"n={int(v):,}" for v in reg["total_stops"]]

yerr = np.vstack([
    np.clip(reg["pct_critical"] - reg["pct_critical_lo"], 0, None),
    np.clip(reg["pct_critical_hi"] - reg["pct_critical"], 0, None),
])

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

bars = axes[0].bar(labels, reg["critical_stops"], color=colors, edgecolor="white", hatch=hatches)
axes[0].set_title("Critical stops per region (count)")
axes[0].set_ylabel("number of critical stops")
axes[0].margins(y=0.15)
annotate_bars(axes[0], bars, n_texts)

bars2 = axes[1].bar(labels, reg["pct_critical"], color=colors, edgecolor="white",
                    hatch=hatches, yerr=yerr, capsize=5, ecolor="#334155")
axes[1].axhline(100 * n_crit / n_total, color="#334155", ls="--", lw=1.2,
                label=f"network average ({100*n_crit/n_total:.1f}%)")
axes[1].set_title("Share of the region's own stops that are critical")
axes[1].set_ylabel("% of the region's stops")
axes[1].margins(y=0.20)
axes[1].legend(fontsize=9)
annotate_bars(axes[1], bars2, n_texts)

fig.suptitle(f"Critical = betweenness in the network-wide top {100*(1-CRITICAL_QUANTILE):.0f}%"
             "   (error bars: 95% Wilson CI)", fontsize=11)
plt.tight_layout()
plt.savefig(FIGURES / "critical_stations_by_region.png", dpi=150)
plt.show()
print("saved ->", FIGURES / "critical_stations_by_region.png")

## 12. Figure 2 - structural failure points by region

Where figure 1 measured *load*, this one measures *structure*: the share of stations in each region that are articulation points, and (when `bridges.csv` is available) how many bridges touch the region per 1,000 stations. A region can carry moderate traffic and still be fragile if its network is a chain of single links, and that's exactly what these two bars reveal.

In [ ]:
has_bridges = "bridges_per_1000_stops" in reg.columns
ncols = 2 if has_bridges else 1
fig, axes = plt.subplots(1, ncols, figsize=(6.5 * ncols, 5))
axes = np.atleast_1d(axes)

b = axes[0].bar(labels, reg["pct_ap"], color=colors, edgecolor="white", hatch=hatches)
axes[0].set_title("Articulation points as % of the region's stops")
axes[0].set_ylabel("% of the region's stops")
axes[0].margins(y=0.15)
annotate_bars(axes[0], b, n_texts)

if has_bridges:
    b2 = axes[1].bar(labels, reg["bridges_per_1000_stops"], color=colors,
                     edgecolor="white", hatch=hatches)
    axes[1].set_title("Bridges touching the region per 1,000 stops")
    axes[1].set_ylabel("bridges per 1,000 stops")
    axes[1].margins(y=0.15)
    annotate_bars(axes[1], b2, n_texts)

plt.tight_layout()
plt.savefig(FIGURES / "ap_and_bridges_by_region.png", dpi=150)
plt.show()
print("saved ->", FIGURES / "ap_and_bridges_by_region.png")

## 13. Figure 3 - how load is distributed within each region

Averages hide the shape of the distribution, so we show two things. **Left**: the mean betweenness per region - how much shortest-path traffic a typical station carries. **Right**: the betweenness distribution within each region on a log scale (zeros dropped, since log(0) is undefined), revealing whether a region has a few extreme hubs or a broad plateau. The dashed line is the global p90 threshold, that is, the cutoff above which a station counts as critical.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

b = axes[0].bar(labels, reg["avg_betweenness"], color=colors, edgecolor="white", hatch=hatches)
axes[0].set_title("Mean betweenness centrality per region")
axes[0].set_ylabel("mean betweenness")
axes[0].margins(y=0.15)
annotate_bars(axes[0], b, n_texts)

order = list(reg["region"])
data = [metrics.loc[(metrics["region"] == r) & (metrics["betweenness"] > 0), "betweenness"].values
        for r in order]
parts = axes[1].boxplot(data, labels=labels, showfliers=False, patch_artist=True)
for patch, c in zip(parts["boxes"], colors):
    patch.set_facecolor(c)
    patch.set_alpha(0.65)
axes[1].set_yscale("log")
axes[1].axhline(max(BTW_THRESHOLD, 1e-12), color="#334155", ls="--", lw=1.2,
                label=f"critical threshold (p{CRITICAL_QUANTILE*100:.0f})")
axes[1].set_title("Betweenness distribution, non-zero stops only (log scale)")
axes[1].set_ylabel("betweenness")
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig(FIGURES / "avg_betweenness_by_region.png", dpi=150)
plt.show()
print("saved ->", FIGURES / "avg_betweenness_by_region.png")

## 14. Figure 4 - the geographic picture

Every station is plotted at its coordinates, colored by region, with the critical stations drawn on top in black. This is a sanity check as much as a result: defining the regions as latitude strips back in step 01 should look like clean horizontal cuts, and the critical stations should trace the inter-city corridors rather than scatter at random. Stations without coordinates are dropped, and the number dropped is printed.

In [ ]:
geo = metrics.dropna(subset=["lat", "lon"])
print(f"stops without coordinates (dropped from the map): {len(metrics) - len(geo):,}")

fig, ax = plt.subplots(figsize=(8, 11))
for r in in_region_order(geo["region"]):
    grp = geo[geo["region"] == r]
    ax.scatter(grp["lon"], grp["lat"], s=2, alpha=0.30, color=region_color(r),
               label=f"{region_label(r)} (n={len(grp):,})")

crit_geo = geo[geo["is_critical"]]
ax.scatter(crit_geo["lon"], crit_geo["lat"], s=14, color="black", alpha=0.70, zorder=5,
           label=f"critical (n={len(crit_geo):,})")

ax.set_title("Stops by region, critical stops in black")
ax.set_xlabel("longitude")
ax.set_ylabel("latitude")
ax.set_aspect(1 / np.cos(np.radians(float(geo["lat"].mean()))))
ax.legend(markerscale=4, fontsize=9, loc="upper left")
plt.tight_layout()
plt.savefig(FIGURES / "stations_map_by_region.png", dpi=150)
plt.show()
print("saved ->", FIGURES / "stations_map_by_region.png")

## 15. Figure 5 - a four-panel regional comparison

The summary figure for the report: the four measures side by side over the same set of regions - percent critical, percent articulation points, average degree (how connected a typical station is), and the share of *the network's total critical stations* found in each region. That last panel is the concentration view: a region that holds a far larger share of the critical stations than of the stations overall is where a nationwide disruption would hurt most, so we draw its share of all stations as a reference marker.

In [ ]:
panels = [
    ("pct_critical", "% critical stops", "% of the region's stops"),
    ("pct_ap", "% articulation points", "% of the region's stops"),
    ("avg_degree", "Mean degree", "neighbouring stops"),
    ("share_of_all_critical", "Share of ALL critical stops", "% of the network's critical stops"),
]

fig, axes = plt.subplots(1, 4, figsize=(19, 5))
for ax, (col, title, ylab) in zip(axes, panels):
    bars = ax.bar(labels, reg[col], color=colors, edgecolor="white", hatch=hatches)
    ax.set_title(title)
    ax.set_ylabel(ylab)
    ax.margins(y=0.18)
    annotate_bars(ax, bars, n_texts, fontsize=8)
    if col == "share_of_all_critical":
        ax.scatter(labels, reg["share_of_all_stops"], color="black", marker="_", s=400,
                   zorder=6, label="share of all stops")
        ax.legend(fontsize=8)

fig.suptitle("Regional comparison of network vulnerability (hatched = small sample)", fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES / "regional_vulnerability_comparison.png", dpi=150)
plt.show()
print("saved ->", FIGURES / "regional_vulnerability_comparison.png")

## 16. Figure 6 - the same comparison by metro area

Repeats figure 5 for the metro breakdown, which is the sharper contrast: four dense urban areas against everything else ("Periphery"). The groups are sorted by their criticality rate, `n` is marked, and any group with fewer stations than `MIN_REGION_N` is hatched, so a spuriously high percentage won't be read as a finding.

In [ ]:
mt = metro_summary.copy()
m_labels = [metro_label(m) for m in mt["metro"]]
m_colors = [metro_color(m) for m in mt["metro"]]
m_hatch = ["//" if f else "" for f in mt["small_sample"]]
m_texts = [f"n={int(v):,}" for v in mt["total_stops"]]

m_yerr = np.vstack([
    np.clip(mt["pct_critical"] - mt["pct_critical_lo"], 0, None),
    np.clip(mt["pct_critical_hi"] - mt["pct_critical"], 0, None),
])

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

b0 = axes[0].bar(m_labels, mt["pct_critical"], color=m_colors, edgecolor="white",
                 hatch=m_hatch, yerr=m_yerr, capsize=4, ecolor="#334155")
axes[0].axhline(100 * n_crit / n_total, color="#334155", ls="--", lw=1.2)
axes[0].set_title("% critical stops (95% Wilson CI)")
axes[0].set_ylabel("% of the area's stops")
axes[0].margins(y=0.20)
annotate_bars(axes[0], b0, m_texts, fontsize=8)

b1 = axes[1].bar(m_labels, mt["pct_ap"], color=m_colors, edgecolor="white", hatch=m_hatch)
axes[1].set_title("% articulation points")
axes[1].set_ylabel("% of the area's stops")
axes[1].margins(y=0.18)
annotate_bars(axes[1], b1, m_texts, fontsize=8)

b2 = axes[2].bar(m_labels, mt["avg_degree"], color=m_colors, edgecolor="white", hatch=m_hatch)
axes[2].set_title("Mean degree")
axes[2].set_ylabel("neighbouring stops")
axes[2].margins(y=0.18)
annotate_bars(axes[2], b2, m_texts, fontsize=8)

for ax in axes:
    ax.tick_params(axis="x", rotation=20)

fig.suptitle("Metropolitan comparison (hatched = small sample)", fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES / "metro_vulnerability_comparison.png", dpi=150)
plt.show()
print("saved ->", FIGURES / "metro_vulnerability_comparison.png")

## 17. Which stations actually drive each region's number

Percentages are abstract; naming the stations makes the result checkable against reality. For each region we list the five critical stations with the highest betweenness, together with whether each is also an articulation point. If these names are familiar inter-city hubs, then the pipeline is measuring something real.

In [ ]:
cols = ["region", "metro", "stop_id", "stop_name", "degree", "betweenness", "is_ap"]
cols = [c for c in cols if c in metrics.columns]

top_by_region = (metrics[metrics["is_critical"]]
                 .sort_values("betweenness", ascending=False)
                 .groupby("region", group_keys=False)
                 .head(5)[cols]
                 .sort_values(["region", "betweenness"], ascending=[True, False]))

top_by_region.to_csv(TABLES / "top_critical_by_region.csv", index=False, encoding="utf-8-sig")
print(top_by_region.to_string(index=False))
print(f"\nsaved -> {TABLES / 'top_critical_by_region.csv'}")

print("\nAll artifacts written by this notebook:")
for p in sorted(STAGE.rglob("*")):
    if p.is_file():
        print("  ", p.relative_to(OUT))

## Conclusions

*(The exact numbers are printed by the cells above and saved to
`outputs/nb/07_regional_comparison/tables/regional_summary.csv`; the statements below describe the pattern
those tables show.)*

1. **Vulnerability isn't spread evenly, but the gap is moderate.** Under the project's definition (critical = top
   decile of network-wide betweenness), the center region has the *lowest* share of critical stations, while the
   north, south, and Jerusalem all sit above the network average of 10%. The direction supports the "periphery is
   more fragile" hypothesis, but the gap is a few percentage points - not an order of magnitude.

2. **The structural signal is stronger than the traffic signal.** The share of stations that are articulation
   points differs between regions by a larger relative factor than the share of critical stations does, with
   Jerusalem clearly high and the center lowest. Dense, tangled networks contain redundancy; sparse networks route
   everything through single stations. This is the more convincing evidence for the regional-inequality claim.

3. **In absolute terms the center still dominates.** It holds far more critical stations than any other region,
   simply because it holds far more stations. A nationwide-disruption analysis should read the counts panel, and a
   fairness analysis should read the percentages panel - they point in opposite directions, and reporting only one
   of them would mislead.

4. **Statistical significance here is almost meaningless; the effect size is not.** With about 30 thousand stations,
   the chi-square p-value is astronomically small, and yet Cramer's V comes out very low. The honest reading: the
   regional differences are *real but weak*. The 95% Wilson intervals in the figures are narrow only because each
   region contains thousands of stations.

5. **No region is a small sample; check the metro table instead.** All four regions hold thousands of stations, so
   none of them is flagged by `MIN_REGION_N`. The flag exists mainly for the metro breakdown and for anyone who
   reruns with a finer geographic split - any hatched bar or row where `small_sample = True` must not be cited as a
   finding.

### Limitations - stated honestly

- **The region labels are latitude strips**, drawn in step 01 from fixed coordinate thresholds, not official
  administrative districts. A station just north of a threshold is assigned to a different region than its actual
  neighbor. Everything said in this notebook inherits that approximation.
- **Betweenness is approximate** (sampling `k` sources in an earlier step) and computed on the largest connected
  component only, so stations in small components get a score of 0 and are never called critical - even though
  being in an isolated, tiny component is itself a kind of fragility.
- **This is a topological analysis.** A station with high betweenness isn't necessarily a station with many riders;
  the graph has no ridership data.
- **There's no causal claim here.** The result says periphery networks are structurally sparser, but not why, and
  not what a specific closure would cost - notebook 05 (robustness) is where removal is actually simulated.